# 17_PrepareBonus: Dataset_bonus -> landmark (cùng định dạng 01_PrepareData)
- Trích landmark: `landmarks/Dataset_bonus/<lop>/<tên>.npz` (lm, hand, fps) như cell 5 của 01.
- Nội suy 120 fps (CUDA, theo thời gian thật): `landmarks_120/Dataset_bonus/<lop>/<tên>.npz` như cell 6 của 01.
- Ghi `manifest_bonus.csv` riêng (cùng cột manifest.csv, `tap` để trống). **Không sửa `manifest.csv`.**
- Chạy lại được: file nào có rồi thì bỏ qua.

In [1]:
#cell 1: cấu hình đường dẫn, kiểm tra CUDA và tên thư mục lớp của Dataset_bonus
import cv2, numpy as np, pandas as pd, torch
from pathlib import Path

assert torch.cuda.is_available(), "Chưa bật CUDA"
dev = "cuda"
BASE = Path(r"C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321")
NGUON = "Dataset_bonus"
SRC = BASE / NGUON
LM = BASE / "landmarks" / NGUON
LM120 = BASE / "landmarks_120" / NGUON
LABEL = {"paper": 0, "rock": 1, "scissors": 2}

print("Dataset_bonus:", SRC.exists(), "| model:", (BASE / "hand_landmarker.task").exists())
lops = sorted(p.name for p in SRC.iterdir() if p.is_dir())
print("Thư mục lớp:", lops)
assert set(l.lower() for l in lops) <= set(LABEL), "Tên thư mục lớp chưa đúng (paper/rock/scissors), sửa tên rồi chạy lại"

Dataset_bonus: True | model: True
Thư mục lớp: ['paper', 'rock', 'scissors']


In [2]:
#cell 2: quét video Dataset_bonus, xem fps / số khung / độ dài, đánh dấu video không phải 60 fps
exts = {".mp4", ".mov", ".avi", ".mkv"}
rows = []
for p in sorted(SRC.rglob("*")):
    if p.suffix.lower() in exts:
        cap = cv2.VideoCapture(str(p)); fps = cap.get(cv2.CAP_PROP_FPS); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); cap.release()
        rows.append(dict(path=p, lop=p.parent.name.lower(), file=p.name, fps=round(fps, 2), khung=n,
                         giay=round(n / fps, 2) if fps else 0, MB=round(p.stat().st_size / 1e6, 2)))
vid = pd.DataFrame(rows)
print(vid.groupby("lop").agg(so_video=("file", "count"), giay_tb=("giay", "mean"), MB=("MB", "sum")).round(2))
print("Phân bố fps:"); print(vid.fps.value_counts())
print("Video khác 60 fps:"); print(vid[vid.fps.sub(60).abs() > 0.5][["lop", "file", "fps", "khung", "giay"]])

          so_video  giay_tb     MB
lop                               
paper           30     1.04  51.37
rock            30     1.03  48.04
scissors        30     1.04  52.46
Phân bố fps:
fps
60.0    90
Name: count, dtype: int64
Video khác 60 fps:
Empty DataFrame
Columns: [lop, file, fps, khung, giay]
Index: []


In [3]:
#cell 3: trích landmark bằng MediaPipe (CPU) như cell 5 của 01, lưu .npz vào landmarks\\Dataset_bonus, bỏ qua file đã có
import mediapipe as mp
from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision

opts = vision.HandLandmarkerOptions(
    base_options=mp_tasks.BaseOptions(model_asset_path=str(BASE / "hand_landmarker.task")),
    running_mode=vision.RunningMode.VIDEO, num_hands=1)

def extract(path):
    det = vision.HandLandmarker.create_from_options(opts)   # tạo mới mỗi video để không giữ trạng thái cũ
    cap = cv2.VideoCapture(str(path)); fps = cap.get(cv2.CAP_PROP_FPS)
    lm, hand, i = [], [], 0
    while True:
        ok, frame = cap.read()
        if not ok: break
        img = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        r = det.detect_for_video(img, int(i * 1000 / fps))
        if r.hand_landmarks:
            lm.append([[p.x, p.y, p.z] for p in r.hand_landmarks[0]])
            hand.append(0 if r.handedness[0][0].category_name == "Left" else 1)
        else:
            lm.append(np.full((21, 3), np.nan)); hand.append(-1)   # mất tay: NaN, nhãn -1
        i += 1
    cap.release(); det.close()
    return np.array(lm, np.float32), np.array(hand, np.int8), fps

stats = []
for r in vid.itertuples():
    f = LM / r.lop / (Path(r.file).stem + ".npz")
    if not f.exists():
        f.parent.mkdir(parents=True, exist_ok=True)
        lm, hand, fps = extract(r.path)
        np.savez_compressed(f, lm=lm, hand=hand, fps=fps)
    else:
        lm = np.load(f)["lm"]
    stats.append((r.lop, np.isnan(lm[:, 0, 0]).mean()))

s = pd.DataFrame(stats, columns=["lop", "ty_le_mat_tay"])
print(s.groupby("lop").agg(so_video=("lop", "count"), mat_tay_tb=("ty_le_mat_tay", "mean")).round(3))
print("Đã lưu tại:", LM)

          so_video  mat_tay_tb
lop                           
paper           30       0.006
rock            30       0.006
scissors        30       0.014
Đã lưu tại: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\landmarks\Dataset_bonus


In [4]:
#cell 4: nội suy landmark lên 120 fps bằng CUDA theo thời gian thật (60 fps = chèn khung giữa như cell 6 của 01), khung mất tay giữ NaN
def up120(lm, hand, fps):
    a = torch.tensor(lm, device=dev); h = torch.tensor(hand, device=dev); T = len(a)
    t = torch.arange(int((T - 1) / fps * 120 + 1e-6) + 1, device=dev) * (fps / 120)   # vị trí khung mới tính theo khung gốc
    i0 = t.floor().long().clamp(max=T - 1); i1 = (i0 + 1).clamp(max=T - 1)
    w = (t - i0).view(-1, 1, 1)
    new = torch.where(w < 1e-6, a[i0], a[i0] + w * (a[i1] - a[i0]))                    # NaN nếu 1 trong 2 khung mất tay
    nh = torch.where((w.view(-1) < 1e-6) | (h[i0] == h[i1]), h[i0], torch.full_like(h[i0], -1))
    return new.cpu().numpy(), nh.cpu().numpy()

rows = []
for f in sorted(LM.rglob("*.npz")):
    d = np.load(f); fps = float(d["fps"]); assert fps > 0, f
    lm, hand = up120(d["lm"], d["hand"], fps)
    o = LM120 / f.parent.name / f.name
    o.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(o, lm=lm, hand=hand, fps=120.0)
    rows.append((f.parent.name, f.name, fps, len(d["lm"]), len(lm), np.isnan(d["lm"][:, 0, 0]).mean()))

r = pd.DataFrame(rows, columns=["lop", "file", "fps_goc", "khung_goc", "khung_120", "ty_le_mat_tay"])
print(r.groupby("lop").agg(so_video=("file", "count"), khung_goc=("khung_goc", "mean"), khung_120=("khung_120", "mean")).round(1))
print("Video mất tay > 10%:"); print(r[r.ty_le_mat_tay > 0.1][["lop", "file", "fps_goc", "ty_le_mat_tay"]])
print("Đã lưu tại:", LM120)

          so_video  khung_goc  khung_120
lop                                     
paper           30       62.6      123.2
rock            30       61.6      121.3
scissors        30       62.7      123.4
Video mất tay > 10%:
         lop                       file    fps_goc  ty_le_mat_tay
22     paper  VID_20261025125210425.npz  60.000018       0.123077
34      rock  VID_20261025002108086.npz  60.000019       0.145161
67  scissors  VID_20261025003208400.npz  60.000019       0.338710
Đã lưu tại: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\landmarks_120\Dataset_bonus


In [5]:
#cell 5: ghi manifest_bonus.csv (cùng cột manifest.csv, tap để trống, KHÔNG sửa manifest.csv) và kiểm tra file thừa
rows = []
for f in sorted(LM120.rglob("*.npz")):
    d = np.load(f)
    rows.append(dict(path=str(f.relative_to(BASE)), nguon=NGUON, lop=f.parent.name,
                     nhan=LABEL[f.parent.name], khung=len(d["lm"]), fps=float(d["fps"]), tap=""))
mb = pd.DataFrame(rows)
mb.to_csv(BASE / "manifest_bonus.csv", index=False)
print(mb.groupby("lop").agg(so_video=("path", "count"), khung_tb=("khung", "mean")).round(1))
print("Tổng:", len(mb), "video |", "video quét được:", len(vid), "| lưu tại:", BASE / "manifest_bonus.csv")
names = {(r.lop, Path(r.file).stem) for r in vid.itertuples()}
thua = [f for f in LM.rglob("*.npz") if (f.parent.name, f.stem) not in names]
print("File thừa trong landmarks\\Dataset_bonus:", thua or "không có")

          so_video  khung_tb
lop                         
paper           30     123.2
rock            30     121.3
scissors        30     123.4
Tổng: 90 video | video quét được: 90 | lưu tại: C:\Users\Acer Nitro 5 Tiger\Downloads\Data_321\manifest_bonus.csv
File thừa trong landmarks\Dataset_bonus: không có
